# ?? IMPORTANT: People-Only Filter Pipeline (Comparison Run)

This notebook executes the full **Leg 2 (w150 sparse)** sequence for the people-only filter experiment (Tasks 1-3).
It is intended to produce comparison data against the canonical baseline, **not** to replace it.

**Dependency Note:**
Consistent with historical canonical runs in this project, `torch`, `torchvision`, and `transformers` are **not pinned** here. Results will reflect whatever versions are currently standard in Google Colab's default environment. Explicitly pinning these ML dependencies has been identified as a deferred gap and is not fixed in this notebook.

# Extract and Train Sparse Embeddings (w150 Dataset)
This notebook performs the GPU-intensive extraction of DINOv3 and ReID embeddings for 4 sampled frames per shot (instead of a single keyframe) on the w150 dataset. It then trains the Graph Transformer to see if this sparse temporal representation improves the clustering/false-positive issue.

In [9]:
import os
import sys
from pathlib import Path
from google.colab import drive
import torch

# ==============================================================================
# Step 1: Mount Drive and set environment variable
# ==============================================================================
print("[STEP 1] Mounting Google Drive...")
drive.mount('/content/drive')
os.environ["DRIVE_ROOT"] = "/content/drive/MyDrive/CCTV-Multiview-Project"
DRIVE_ROOT = Path(os.environ["DRIVE_ROOT"])


[STEP 1] Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# ==============================================================================
# Step 2: Verify GPU
# ==============================================================================
print("\n[STEP 2] Verifying GPU...")
if not torch.cuda.is_available():
    print("[FAIL] GPU is not available! Please change the runtime type to T4/A100 GPU and restart.")
    sys.exit(1)
print(f"[PASS] GPU detected: {torch.cuda.get_device_name(0)}")



[STEP 2] Verifying GPU...
[PASS] GPU detected: Tesla T4


In [11]:
# ==============================================================================
# Step 3: Setup Environment
# ==============================================================================
repo_dir = "/content/cctv-multiview-summarization"
if not os.path.exists(repo_dir):
    print(f"[INFO] Cloning repository to {repo_dir}...")
    !git clone https://github.com/Gautam-Shah306/cctv-multiview-summarization.git {repo_dir}

os.chdir(repo_dir)
!git fetch origin
!git checkout feature/stage1-object-detection
!git pull origin feature/stage1-object-detection
!pip install -q -r requirements-colab.txt

print("\n[INFO] Installing PyTorch Geometric...")
!pip install -q torch-geometric


remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 9 (delta 6), reused 9 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 6.94 KiB | 1.73 MiB/s, done.
From https://github.com/Gautam-Shah306/cctv-multiview-summarization
   e6e7b89..b5391ce  feature/stage1-object-detection -> origin/feature/stage1-object-detection
Already on 'feature/stage1-object-detection'
Your branch is behind 'origin/feature/stage1-object-detection' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/Gautam-Shah306/cctv-multiview-summarization
 * branch            feature/stage1-object-detection -> FETCH_HEAD
Updating e6e7b89..b5391ce
Fast-forward
 notebooks/extract_and_train_sparse.ipynb |  51 +++++-
 scratch/infer_and_evaluate_sparse.py     | 260 +++++++++++++++++++++++++++++++
 scratch_build_ipynb_sparse.py            | 127 +++++++++++++++
 src/as

# ==============================================================================
# Step 3.5: Run Upstream People-Only Filters (w150)
# ==============================================================================
# These CPU steps are executed here to ensure an atomic, end-to-end Leg 2 run 
# inside a single Colab session, incorporating the new Task 1-2 filters.

In [ ]:
print("\n[STEP 3.5] Running Upstream Filtering and Pair Generation...")
!python -m src.keyframe_selection --output-suffix w150 --shot-window-sec 6.0
!python -m src.generate_training_pairs --suffix w150

In [ ]:
# ==============================================================================
# Step 4: Extract Sparse-Sampled Embeddings
# ==============================================================================
print("\n[STEP 4] Extracting Sparse-Sampled Embeddings...")
!python -m src.extract_sparse_embeddings


In [ ]:
# ==============================================================================
# Step 5: Run 5-Fold CV Training (Sparse w150 dataset)
# ==============================================================================
print("\n[STEP 5] Running Graph Transformer 5-Fold Training on Sparse Data...")
!python -m src.train_graph_transformer_w150_sparse


In [12]:
print("\n[STEP 6] Persisting Models & Sparse Embeddings to Google Drive...")
import shutil
from pathlib import Path

# Copy models (Models inherently have new fold outputs, we can keep them or add _people_only)
# Let's add _people_only to model names to prevent overwrite
for fold in range(1, 6):
    local_model = Path(f"models/graph_transformer_w150_sparse_fold{fold}.pt")
    drive_model = DRIVE_ROOT / "models" / f"graph_transformer_w150_sparse_fold{fold}_people_only.pt"
    drive_model.parent.mkdir(parents=True, exist_ok=True)

    if local_model.exists():
        shutil.copy2(local_model, drive_model)
        size_mb = drive_model.stat().st_size / (1024 * 1024)
        print(f"[PASS] Fold {fold} Model successfully copied to Drive: {drive_model} ({size_mb:.2f} MB)")

# Copy newly extracted embeddings back to Drive (append _people_only)
dino_sparse = Path("data_manifests/training_features_dino_w150_sparse.npz")
reid_sparse = Path("data_manifests/training_features_reid_w150_sparse.npz")

for f in [dino_sparse, reid_sparse]:
    if f.exists():
        new_name = f.stem + "_people_only" + f.suffix
        drive_target = DRIVE_ROOT / "data_manifests" / new_name
        drive_target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, drive_target)
        size_mb = drive_target.stat().st_size / (1024 * 1024)
        print(f"[PASS] Sparse embeddings successfully copied to Drive: {drive_target.name} ({size_mb:.2f} MB)")



[STEP 6] Persisting Models & Sparse Embeddings to Google Drive...
[PASS] Fold 1 Model successfully copied to Drive: /content/drive/MyDrive/CCTV-Multiview-Project/models/graph_transformer_w150_sparse_fold1.pt (0.12 MB)
[PASS] Fold 2 Model successfully copied to Drive: /content/drive/MyDrive/CCTV-Multiview-Project/models/graph_transformer_w150_sparse_fold2.pt (0.12 MB)
[PASS] Fold 3 Model successfully copied to Drive: /content/drive/MyDrive/CCTV-Multiview-Project/models/graph_transformer_w150_sparse_fold3.pt (0.12 MB)
[PASS] Fold 4 Model successfully copied to Drive: /content/drive/MyDrive/CCTV-Multiview-Project/models/graph_transformer_w150_sparse_fold4.pt (0.12 MB)
[PASS] Fold 5 Model successfully copied to Drive: /content/drive/MyDrive/CCTV-Multiview-Project/models/graph_transformer_w150_sparse_fold5.pt (0.12 MB)
[PASS] Sparse embeddings successfully copied to Drive: /content/drive/MyDrive/CCTV-Multiview-Project/data_manifests/training_features_dino_w150_sparse.npz (0.14 MB)
[PASS] S

In [13]:
import sys
from unittest.mock import patch
from src import run_graph_transformer_inference

print("\n[STEP 7] Running Final Inference (Video generation mocked for speed)...")
# We mock generate_video because rendering 6 videos is slow and unnecessary for a pure numerical comparison run.
with patch('src.run_graph_transformer_inference.generate_video', lambda csv, out: print(f"[MOCK] Skipped generating video for {csv}")):
    run_graph_transformer_inference.main()



[STEP 7] Running Final Inference and Generating Summary Videos...
[INFO] Running on cuda...
[INFO] Found 81 unique nodes (shots).
[INFO] DINOv3 PCA (384 -> 64) Explained Variance: 0.9937
[INFO] ReID PCA (512 -> 64) Explained Variance: 0.9985
[INFO] New node feature matrix shape: torch.Size([81, 133]) (Dims: 133)
[INFO] Graph built with 612 unique undirected edges.
[INFO] Retraining model on ALL labeled training pairs (612 edges) for final inference...
[INFO] Predicting all possible pairwise edges for 81 nodes...
\n--- GRAPH TRANSFORMER (150-frame) CLUSTERS ---
Total nodes: 81
Total clusters: 10
Cross-view clusters: 1
[DONE] Wrote final_summary_graph_transformer.csv with 10 sequences.
\n[INFO] Generating Per-View Summaries...
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/indexes/base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._

In [ ]:
print("\n[STEP 8] Copying ALL generated CSVs to Drive safely...")
import glob
import shutil
from pathlib import Path

# We grab all manifests (inference outputs + upstream outputs)
artifacts = glob.glob("data_manifests/*.csv") + glob.glob("data_manifests/*.mp4")
for artifact in artifacts:
    local_path = Path(artifact)
    
    # We want to sync inference outputs AND the new keyframe/training pairs
    if "graph_transformer" in artifact or "ruleBased" in artifact or "w150" in artifact:
        # Inject _people_only into the filename to prevent overwriting canonicals on Drive
        new_name = local_path.stem + "_people_only" + local_path.suffix
        drive_path = DRIVE_ROOT / "data_manifests" / new_name
        drive_path.parent.mkdir(parents=True, exist_ok=True)
        
        shutil.copy2(local_path, drive_path)
        print(f"[PASS] Copied {local_path.name} -> {new_name} on Drive.")
